# Building an Image Classifier with PyTorch (Fashion MNIST)

Reproduction of the "Building an Image Classifier with PyTorch" section of Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, chapter 10. Each code cell below is one generated script from `single_prompt/`, run in order.

Generated script: 01_setup.py

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", device)

Generated script: 02_data.py

In [ ]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55_000, 5_000])

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=32)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32)

class_names = train_and_valid_data.classes

X_sample, y_sample = train_data[0]
print("Sample shape:", X_sample.shape, "| dtype:", X_sample.dtype)
print("Sample label:", y_sample, "->", class_names[y_sample])

Generated script: 03_model.py

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, n_inputs, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(42)
model = ImageClassifier(n_inputs=28 * 28, n_hidden1=300, n_hidden2=100,
                        n_classes=10).to(device)
xentropy = nn.CrossEntropyLoss()
print(model)

Generated script: 04_train.py

In [ ]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, loss_fn, metric, train_loader, valid_loader,
           n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history


learning_rate = 0.1
n_epochs = 5
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
history = train2(model, optimizer, xentropy, accuracy, train_loader,
                 valid_loader, n_epochs)

Generated script: 05_plot_accuracy.py

In [ ]:
epochs = np.arange(1, n_epochs + 1)
plt.figure(figsize=(7, 4))
plt.plot(epochs, history["train_metrics"], "b.-", label="Training accuracy")
plt.plot(epochs, history["valid_metrics"], "r.-", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fashion MNIST MLP: accuracy per epoch")
plt.xticks(epochs)
plt.grid(True)
plt.legend()
plt.show()

Generated script: 06_predict.py

In [ ]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new, y_new = X_new[:3].to(device), y_new[:3]
with torch.no_grad():
    y_pred_logits = model(X_new)
y_pred = y_pred_logits.argmax(dim=1).cpu()
print("Predicted:", [class_names[i] for i in y_pred])
print("True:     ", [class_names[i] for i in y_new])

y_proba = F.softmax(y_pred_logits, dim=1).cpu()
print("Probabilities:\n", y_proba.round(decimals=3))

y_top4_logits, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_pred_logits, dim=1).gather(1, y_top4_indices).cpu()
for i in range(3):
    top4 = ", ".join(f"{class_names[j]} ({p:.3f})"
                     for j, p in zip(y_top4_indices[i].cpu(), y_top4_probas[i]))
    print(f"Image {i}: top-4 = {top4}")

Generated script: 07_test_eval.py

In [ ]:
test_acc = evaluate_tm(model, test_loader, accuracy)
print(f"Test accuracy: {test_acc.item():.4f}")

Generated script: 08_save_load.py

In [ ]:
model_data = {
    "model_state_dict": model.state_dict(),
    "model_hyperparameters": {"n_inputs": 28 * 28, "n_hidden1": 300,
                              "n_hidden2": 100, "n_classes": 10},
}
torch.save(model_data, "my_fashion_mnist.pt")

loaded_data = torch.load("my_fashion_mnist.pt", weights_only=True)
new_model = ImageClassifier(**loaded_data["model_hyperparameters"]).to(device)
new_model.load_state_dict(loaded_data["model_state_dict"])
new_model.eval()

with torch.no_grad():
    same = torch.allclose(model(X_new), new_model(X_new))
print("Reloaded model gives identical predictions:", same)

Generated script: 09_optuna.py

In [ ]:
def objective(trial, train_loader, valid_loader, n_epochs=3):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    n_hidden = trial.suggest_int("n_hidden", 20, 300)
    model = ImageClassifier(n_inputs=28 * 28, n_hidden1=n_hidden,
                            n_hidden2=n_hidden, n_classes=10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
    best_validation_accuracy = 0.0
    for epoch in range(n_epochs):
        history = train2(model, optimizer, xentropy, accuracy, train_loader,
                         valid_loader, n_epochs=1)
        validation_accuracy = max(history["valid_metrics"])
        best_validation_accuracy = max(best_validation_accuracy,
                                       validation_accuracy)
        trial.report(validation_accuracy, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return best_validation_accuracy


torch.manual_seed(42)
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner()
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(lambda trial: objective(trial, train_loader, valid_loader),
               n_trials=2)
print("Best params:", study.best_params)
print("Best validation accuracy:", study.best_value)